# Module 3.1: UniFi Drive Onboarding, RAID 1 Storage Pool, Btrfs Snapshots & NFS/SMB Shares

## UniFi Storage & Enterprise Networking (Chapter 3) - Step 3: NAS Storage Pool & File Sharing

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Initialize **UniFi OS & UniFi Drive** on your `UNAS 2` (`10.0.0.202`) as a standalone NAS appliance (no UniFi Gateway required).
2. Provision both `4 TB UACC-HDD-S-4TB` drives into a **Basic Protection (RAID 1)** storage pool utilizing **Btrfs** checksumming for silent bit-rot detection and single-drive fault tolerance (`~3.64 TiB` usable).
3. Architect clean **Shared Drives** (`HomelabBackups`, `MediaArchive`, and optional `TimeMachine`) separated by access pattern, permissions, and snapshot retention policy.
4. Configure automated **Btrfs Local Snapshots** to guard against accidental deletion (`rm -rf`) or ransomware encryption.
5. Enable **NFS** (whitelisting strictly `10.0.0.200` and `10.0.0.201`) and **SMB** file-sharing services for both Linux server nodes and desktop workstations.

## 3.1.1 Conceptual Background: Inside UniFi Drive's Storage Stack (RAID 1 + Btrfs + Snapshots)

Unlike a raw Linux `ext4` partition on a single USB SSD (which we built on `homelab-pi` in Chapter 1), the **UniFi UNAS 2** runs **UniFi OS + UniFi Drive**—a purpose-built Linux storage appliance combining three defensive layers:

```mermaid
flowchart TB
    subgraph UNAS2 ["UniFi UNAS 2 Storage Stack (10.0.0.202)"]
        Bay1["Bay 1: UACC-HDD-S-4TB<br>4 TB 5400 RPM SATA"]
        Bay2["Bay 2: UACC-HDD-S-4TB<br>4 TB 5400 RPM SATA"]
        RAID1["Layer 1: Linux Software RAID 1 (Mirror)<br>1-Drive Fault Tolerance | ~3.64 TiB Usable"]
        Btrfs["Layer 2: LVM + Btrfs Filesystem<br>Copy-on-Write (CoW), Checksumming & Zero-Cost Snapshots"]
        
        subgraph Shares ["Layer 3: UniFi Drive Shared Drives"]
            S1["HomelabBackups<br>(NFS R/W to .200 & .201)<br>Daily + Weekly Snapshots"]
            S2["MediaArchive<br>(NFS R/O to .200, R/W to .201 + SMB)<br>Weekly Snapshots"]
            S3["TimeMachine (Optional)<br>(SMB w/ Quota for macOS)"]
        end
    end
    
    Bay1 --> RAID1
    Bay2 --> RAID1
    RAID1 --> Btrfs
    Btrfs --> S1
    Btrfs --> S2
    Btrfs --> S3
```

### 1. Why Btrfs Copy-on-Write (CoW) Snapshots Are a Game-Changer
Remember the classic homelab mistake: accidentally running `rsync --delete` with a typo in the source path, or having a workstation compromised by ransomware that encrypts mounted network shares.
- **RAID 1 alone will NOT save you from accidental deletion or ransomware!** A RAID 1 mirror dutifully deletes or encrypts the file on *both* hard drives in milliseconds.
- **Btrfs Local Snapshots WILL save you!** Because Btrfs is Copy-on-Write, taking a daily snapshot of `HomelabBackups` or `MediaArchive` takes **<1 second** and consumes **almost zero extra disk space** (only changed blocks consume additional space). If a file is accidentally wiped or overwritten, you roll back to yesterday's read-only snapshot directly inside the UniFi Drive web UI!

### 2. Recommended Shared Drive Architecture

| Shared Drive Name | Primary Purpose | NFS Client Whitelist & Mode | SMB Access | Btrfs Snapshot Schedule |
| :--- | :--- | :--- | :---: | :--- |
| **`HomelabBackups`** | Automated nightly Immich/Paperless/Mail-Archiver PostgreSQL dumps, Immich library mirror, Pi-hole/Uptime Kuma/UniFi configs | `10.0.0.200` (`homelab-pi`: **R/W**)<br>`10.0.0.201` (`homelab-gpu`: **R/W**) | Admin only (R/O recommended) | **Daily (keep 7) + Weekly (keep 4)** |
| **`MediaArchive`** | University OneDrive archive, Paperless-ngx PDFs, Immich External Library, workstation shares | `10.0.0.201` (`homelab-gpu`: **R/W**)<br>*(Not needed on `homelab-pi`!)* | Workstations (R/W) | **Weekly (keep 4)** |
| **`TimeMachine`** *(Optional)* | Wireless/wired macOS Time Machine laptop backups | Disabled (SMB only) | macOS Time Machine (cap quota e.g. `1 TB`) | Off *(Time Machine manages its own APFS sparsebundle versions)* |

> 💡 **Why Separate `HomelabBackups` from `MediaArchive`?** Never allow everyday Windows/macOS laptops with read/write SMB access to write directly into your server backup folder! Keeping `HomelabBackups` dedicated strictly to `homelab-pi` and `homelab-gpu` via IP-restricted NFS isolates your disaster-recovery backups from accidental user edits or desktop malware.

## 3.1.2 Step-by-Step Laboratory Guide

### Task 3.1.1: Initialize UniFi OS & Provision the 4 TB RAID 1 Storage Pool
1. Open your browser and navigate to your `UNAS 2`'s reserved static IP:
   ```text
   https://10.0.0.202
   ```
   *(Alternatively, open the **UniFi** mobile app near the `UNAS 2` to onboard via Bluetooth 4.1—both paths lead to the identical UniFi OS setup wizard.)*
2. Complete the **UniFi OS Initial Setup**:
   - **Console Name:** `homelab-nas`
   - **Administrator Account:** Sign in with your Ubiquiti UI account (enables optional `unifi.ui.com` management & mobile UniFi Drive app) OR choose **Local Setup** if you prefer strict LAN + Tailscale-only administration.
   - Apply any pending **UniFi OS / UniFi Drive** firmware updates when prompted so you have the latest NFS, Btrfs snapshot, and Cloud Backup features.
3. Open the **UniFi Drive** application $\rightarrow$ navigate to **Storage** in the left sidebar:
   - Confirm both **Bay 1** and **Bay 2** detect your **`4 TB` `UACC-HDD-S-4TB`** drives as **Healthy** (`5400 RPM`, SATA III, normal temperature `~30°C–42°C`).
   - Select **Basic Protection (RAID 1)** as your storage pool protection mode and confirm creation.
   - Verify usable capacity reports approximately **`3.6 TB` (`~3.64 TiB`)**.
   - *(Note: Initial RAID 1 background parity/mirror synchronization across 4 TB drives takes roughly 5–8 hours. You can immediately create Shared Drives and configure NFS while synchronization runs cleanly in the background—just hold off on running disk speed benchmarks in Module 3.2 until sync reaches 100%!)*

### Task 3.1.2: Create Shared Drives, Schedule Btrfs Snapshots & Enable NFS/SMB
1. In **UniFi Drive**, navigate to **Shared Drives** $\rightarrow$ click **Create Shared Drive** twice to create:
   - **`HomelabBackups`** (No size quota, or e.g. `2 TB` soft cap depending on your Immich growth rate)
   - **`MediaArchive`** (For bulk media, Immich External Libraries, and workstation shares)
   - *(Optional)* **`TimeMachine`** — toggle **Time Machine Support** on and assign a hard quota (e.g., `1 TB`) so macOS backups never fill the entire 4 TB array!
2. **Enable Automated Btrfs Local Snapshots:**
   - Select **`HomelabBackups`** $\rightarrow$ **Settings / Snapshots** $\rightarrow$ enable scheduled snapshots: **Daily (retain 7)** and **Weekly (retain 4)**.
   - Select **`MediaArchive`** $\rightarrow$ enable **Weekly (retain 4)** snapshots.
3. **Enable File Sharing Protocols (`NFS` + `SMB`):**
   - Navigate to **UniFi Drive $\rightarrow$ Settings $\rightarrow$ Services** (or **File Services**).
   - **Enable SMB:** Create a dedicated non-admin UniFi Drive user credential (or use your local NAS user) with access to `MediaArchive` (and `TimeMachine` if created). Note the SMB connection string displayed (e.g., `smb://10.0.0.202/MediaArchive`).
   - **Enable NFS:** Toggle **NFS** on, then click **Add NFS Connection** (or configure per-share NFS permissions) to whitelist your two Linux nodes strictly by static LAN IP:
     - **Client IP `10.0.0.200`** (`homelab-pi` — Tier-0 Core):
       - `HomelabBackups` $\rightarrow$ **Read/Write** *(for nightly Pi-hole & Uptime Kuma config backups; `homelab-pi` does not need `MediaArchive` mounted!)*
     - **Client IP `10.0.0.201`** (`homelab-gpu` — Unified App & Immich Server):
       - `HomelabBackups` $\rightarrow$ **Read/Write**
       - `MediaArchive` $\rightarrow$ **Read/Write** *(FileBrowser & Paperless-ngx read/write here, while Immich mounts it `:ro` inside Docker for safety!)*
   - Record the exact NFS mount path displayed by UniFi Drive—by default UniFi OS exports shared drives at:
     ```text
     10.0.0.202:/var/nfs/shared/HomelabBackups
     10.0.0.202:/var/nfs/shared/MediaArchive
     ```

### 🖼️ Task 3.1.2 Verification: UniFi Drive Healthy RAID 1 Pool & NFS/SMB Configuration

**Verification Objective:** Capture a screenshot of the **UniFi Drive Storage Dashboard** (`https://10.0.0.202`) showing both `4 TB` drives healthy in **Basic Protection (RAID 1)**, along with your `HomelabBackups` and `MediaArchive` shared drives and NFS enabled for `10.0.0.200` and `10.0.0.201`.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] `UNAS 2` (`10.0.0.202`) initialized with UniFi OS & latest UniFi Drive firmware.
- [ ] Both `UACC-HDD-S-4TB` drives healthy and configured in **Basic Protection (RAID 1)** (`~3.64 TiB` usable).
- [ ] `HomelabBackups` and `MediaArchive` shared drives created with automated **Btrfs Local Snapshots** scheduled.
- [ ] **NFS** enabled and strictly IP-whitelisted to `10.0.0.200` (`homelab-pi`) and `10.0.0.201` (`homelab-gpu`), and **SMB** enabled for workstation access.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 3.2: Persistent Linux NFS Client Mounts (`fstab` Automount) & Multi-Gig Benchmarking](./03_2_nfs_client_mounts_and_25gbe_benchmark.ipynb).